# Quantum-Classical Pipeline for CML Drug Selection
## Hybrid QAOA Docking + ADMET Profiling — Combined Notebook

Run every cell **once** from top to bottom.  Only edit the `pdb_path` in **Cell 4** to point to your AlphaFold PDB file.

In [ ]:
# Install all dependencies at once
!pip install rdkit Bio cudaq admet_ai torch argparse

In [ ]:
import os, math, json
import numpy as np
import networkx as nx
from rdkit import Chem, RDConfig
from rdkit.Chem import AllChem, ChemicalFeatures, rdchem, Descriptors, Lipinski, Crippen
from Bio.PDB import PDBParser, NeighborSearch
import cudaq
from cudaq import spin
import itertools


def rdkit_ligand_features(mol, confId=0, families=None):
    fdef = os.path.join(RDConfig.RDDataDir, "BaseFeatures.fdef")
    factory = ChemicalFeatures.BuildFeatureFactory(fdef)
    feats = []
    for f in factory.GetFeaturesForMol(mol, confId=confId):
        fam = f.GetFamily()
        if (families is None) or (fam in families):
            feats.append({
                "id":   ("L", len(feats)),
                "type": fam,
                "pos":  np.array([f.GetPos().x, f.GetPos().y, f.GetPos().z])
            })
    return feats


def find_abl1_atp_site(pdb_path):
    """
    Auto-detect the ABL1 ATP binding site centre.
    Priority: (1) ABL1-canonical residue numbers K271/E286/T315/D381/F382/G383,
              (2) DFG motif scan (Asp-Phe-Gly),
              (3) P-loop GxGxxG scan,
              (4) whole-structure centroid.
    """
    structure = PDBParser(QUIET=True).get_structure("prot", pdb_path)
    residues  = [r for r in structure.get_residues() if r.get_id()[0] == ' ']
    res_names = [r.get_resname() for r in residues]
    anchor    = []

    # 1. ABL1-canonical residue numbers
    for r in residues:
        if r.get_id()[1] in {271, 286, 290, 315, 381, 382, 383} and 'CA' in r:
            anchor.append(r['CA'].coord.copy())

    # 2. DFG motif
    for i in range(len(res_names) - 2):
        if res_names[i]=='ASP' and res_names[i+1]=='PHE' and res_names[i+2]=='GLY':
            for r in residues[i:i+3]:
                if 'CA' in r: anchor.append(r['CA'].coord.copy())
            break

    # 3. P-loop GxGxxG
    for i in range(len(res_names) - 5):
        if res_names[i]=='GLY' and res_names[i+2]=='GLY' and res_names[i+5]=='GLY':
            for r in residues[i:i+6]:
                if 'CA' in r: anchor.append(r['CA'].coord.copy())
            break

    if len(anchor) >= 2:
        centre = np.array(anchor).mean(axis=0)
        print(f"[find_abl1_atp_site] Centre: {centre.round(2)}  ({len(anchor)} anchors)")
        return centre

    # 4. Fallback
    all_coords = np.array([a.coord for r in residues for a in r.get_atoms()
                           if a.element not in ('H', None)])
    centre = all_coords.mean(axis=0)
    print(f"[find_abl1_atp_site] WARNING: no canonical anchors; using centroid {centre.round(2)}")
    return centre


def protein_features_from_pdb(pdb_path, site_center=None, site_radius=10.0):
    """Return receptor pharmacophore points within the binding-site sphere."""
    structure = PDBParser(QUIET=True).get_structure("prot", pdb_path)
    atoms = [a for a in structure.get_atoms() if a.element != "H"]

    if site_center is None:
        site_center = np.array([a.coord for a in atoms]).mean(0)

    feats = []
    for a in atoms:
        p = a.coord
        if np.linalg.norm(p - site_center) > site_radius:
            continue
        res     = a.get_parent()
        resname = res.get_resname().strip()
        aname   = a.get_name().strip()

        # H-bond acceptor (backbone + sidechain O, one feature per O atom)
        if aname == "O":
            feats.append({"id":("R",len(feats)),"type":"Acceptor","pos":p.copy()})

        # H-bond donor (N atoms, mutually exclusive with acceptor check above)
        elif aname in ("N","NE","NE2","ND2","NZ"):
            feats.append({"id":("R",len(feats)),"type":"Donor","pos":p.copy()})

        # Positive ionisable
        if resname in ("LYS","ARG") and aname in ("NZ","CZ","NE","NH1","NH2"):
            feats.append({"id":("R",len(feats)),"type":"PosIonizable","pos":p.copy()})

        # Negative ionisable
        if resname in ("ASP","GLU") and aname.startswith("O"):
            feats.append({"id":("R",len(feats)),"type":"NegIonizable","pos":p.copy()})

        # Aromatic ring atoms  (PHE, TYR, TRP, HIS)
        _aromatic = {
            "PHE":{"CG","CD1","CD2","CE1","CE2","CZ"},
            "TYR":{"CG","CD1","CD2","CE1","CE2","CZ"},
            "TRP":{"CD2","CE2","CE3","CZ2","CZ3","CH2"},
            "HIS":{"CG","ND1","CD2","CE1","NE2"},
        }
        if resname in _aromatic and aname in _aromatic[resname]:
            feats.append({"id":("R",len(feats)),"type":"Aromatic","pos":p.copy()})

        # Hydrophobic centres  (sidechain carbons of apolar residues)
        _hydrophobic = {"LEU","VAL","ILE","MET","PHE","TRP","ALA","PRO"}
        if resname in _hydrophobic and aname.startswith("C") and aname not in ("C","CA"):
            feats.append({"id":("R",len(feats)),"type":"Hydrophobe","pos":p.copy()})

    return feats


COMPLEMENT = {
    ("Donor","Acceptor"), ("Acceptor","Donor"),
    ("PosIonizable","NegIonizable"), ("NegIonizable","PosIonizable"),
    ("Aromatic","Aromatic"),
    ("Hydrophobe","Hydrophobe"),
}


def in_contact_window(tL, tR, dist):
    if {tL,tR} == {"Donor","Acceptor"}:            return 1.6 <= dist <= 3.3
    if {tL,tR} == {"PosIonizable","NegIonizable"}:  return 2.0 <= dist <= 5.0
    if tL == tR == "Aromatic":                       return 3.5 <= dist <= 6.0
    if tL == tR == "Hydrophobe":                     return 3.0 <= dist <= 6.5
    return False


def build_BIG(lig_feats, rec_feats, eps_pair=0.6, max_candidates_per_lig=15):
    nodes = []
    for i, lf in enumerate(lig_feats):
        candidates = []
        for j, rf in enumerate(rec_feats):
            if (lf["type"], rf["type"]) not in COMPLEMENT:
                continue
            d = np.linalg.norm(lf["pos"] - rf["pos"])
            if in_contact_window(lf["type"], rf["type"], d):
                candidates.append((d, i, j))
        candidates.sort()
        for _, i2, j in candidates[:max_candidates_per_lig]:
            nodes.append((i2, j))

    G = nx.Graph()
    G.add_nodes_from(nodes)
    for a in range(len(nodes)):
        i, j  = nodes[a]
        xi,yj = lig_feats[i]["pos"], rec_feats[j]["pos"]
        for b in range(a+1, len(nodes)):
            k, m  = nodes[b]
            if (i==k) or (j==m): continue
            xk,ym = lig_feats[k]["pos"], rec_feats[m]["pos"]
            if abs(np.linalg.norm(xi-xk) - np.linalg.norm(yj-ym)) <= eps_pair:
                G.add_edge((i,j),(k,m))
    return G


def kabsch(P, Q):
    Pc=P.mean(0); Qc=Q.mean(0)
    H = (P-Pc).T @ (Q-Qc)
    U,S,Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1,:] *= -1
        R = Vt.T @ U.T
    return R, Qc - R@Pc


def pose_from_clique(clique, lig_feats, rec_feats):
    P = np.array([lig_feats[i]["pos"] for (i,_) in clique])
    Q = np.array([rec_feats[j]["pos"] for (_,j) in clique])
    R, t = kabsch(P, Q)
    rms  = np.sqrt(((Q-(P@R.T+t))**2).sum(axis=1).mean())
    return R, t, rms


def ham_clique(penalty, nodes, weights, non_edges) -> cudaq.SpinOperator:
    spin_ham = 0
    for wt, node in zip(weights, nodes):
        spin_ham += 0.5*wt*spin.z(node)
        spin_ham -= 0.5*wt*spin.i(node)
    for ne in non_edges:
        u, v = ne[0], ne[1]
        spin_ham += penalty/4.0*(spin.z(u)*spin.z(v) - spin.z(u) - spin.z(v)
                                  + spin.i(u)*spin.i(v))
    return spin_ham


def term_coefficients(ham: cudaq.SpinOperator) -> list[complex]:
    return [t.evaluate_coefficient() for t in ham]


def term_words(ham: cudaq.SpinOperator, qubit_num: int) -> list[str]:
    return [t.get_pauli_word(qubit_num) for t in ham]


@cudaq.kernel
def dc_qaoa(qubit_num:int, num_layers:int, thetas:list[float],
            coef:list[complex], words:list[cudaq.pauli_word]):
    qubits = cudaq.qvector(qubit_num)
    h(qubits)
    count = 0
    for p in range(num_layers):
        for i in range(len(coef)):
            exp_pauli(thetas[count]*coef[i].real, qubits, words[i])
            count += 1
        for j in range(qubit_num):
            rx(thetas[count], qubits[j])
            count += 1
        for k in range(qubit_num):
            ry(thetas[count], qubits[k])
            count += 1

def lipinski_ok(mol):
    return (Descriptors.MolWt(mol)       <  500 and
            Crippen.MolLogP(mol)         <  5   and
            Lipinski.NumHDonors(mol)     <= 5   and
            Lipinski.NumHAcceptors(mol)  <= 10)

In [ ]:
import torch
from argparse import Namespace
from admet_ai import ADMETModel

torch.serialization.add_safe_globals([Namespace])

## 1. Candidate Ligands

In [ ]:
# Eight FDA-approved/candidate TKI drugs for CML targeting BCR-ABL1
# SMILES sourced from PubChem
CANDIDATE_LIGANDS = {
    "Imatinib":   "CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(=N3)C)N4CCN(CC4)CCO",
    "Dasatinib":  "CC1=C(C=C(C=C1)C(=O)NC2=CC(=C(C=C2)CN3CCN(CC3)C)C(F)(F)F)C#CC4=CN=C5N4N=CC=C5",
    "Nilotinib":  "C1CN(C[C@@H]1O)C2=C(C=C(C=N2)C(=O)NC3=CC=C(C=C3)OC(F)(F)Cl)C4=CC=NN4",
    "Bosutinib":  "CNC(=O)C1=CC=CC=C1SC2=CC3=C(C=C2)C(=NN3)/C=C/C4=CC=CC=N4",
    "Ponatinib":  "CC1=C(C=C(C=C1)C(=O)NC2=CC(=CC(=C2)C(F)(F)F)N3C=C(N=C3)C)NC4=NC=CC(=N4)C5=CN=CC=C5",
    "Asciminib":  "CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5",
    "Bafetinib":  "CC1=C(C=C(C=C1)NC(=O)C2=CC(=C(C=C2)CN3CC[C@@H](C3)N(C)C)C(F)(F)F)NC4=NC=CC(=N4)C5=CN=CN=C5",
    "Axitinib":   "CN1CCN(CC1)CCCOC2=C(C=C3C(=C2)N=CC(=C3NC4=CC(=C(C=C4Cl)Cl)OC)C#N)OC",
}

## 2. Protein Setup — Auto-detect ABL1 ATP Binding Site

In [ ]:
# ── Protein setup (run once; all ligands share the same receptor) ────────────
pdb_path = '/content/unrelaxed_model_1_pred_0.pdb'  # ← Replace with your AlphaFold PDB

binding_site_center = find_abl1_atp_site(pdb_path)
binding_site_radius = 10.0  # Å — covers the full ABL1 ATP binding cleft

rec_feats = protein_features_from_pdb(
    pdb_path, site_center=binding_site_center, site_radius=binding_site_radius
)
print(f"Receptor pharmacophores in binding site: {len(rec_feats)}")

## 3. QAOA Docking — All Ligands

In [ ]:
# ── QAOA Docking Loop ────────────────────────────────────────────────────────
binding_scores, ligand_smiles_list, ligand_names_list = [], [], []
NUM_LAYERS = 3  # circuit depth (experimentally chosen)

for drug_name, smiles in CANDIDATE_LIGANDS.items():
    print(f"\n{'='*60}")
    print(f"  Docking: {drug_name}")
    print(f"{'='*60}")

    # Ligand 3-D conformer + pharmacophore features
    lig = Chem.AddHs(Chem.MolFromSmiles(smiles))
    AllChem.EmbedMolecule(lig, AllChem.ETKDG())
    AllChem.MMFFOptimizeMolecule(lig)
    lig_feats = rdkit_ligand_features(lig, confId=0, families=None)
    print(f"  Ligand pharmacophores : {len(lig_feats)}")

    # Build Binding Interaction Graph
    G = build_BIG(lig_feats, rec_feats, eps_pair=0.6, max_candidates_per_lig=15)
    print(f"  BIG — nodes: {G.number_of_nodes()}  edges: {G.number_of_edges()}")

    if G.number_of_nodes() == 0:
        print(f"  WARNING: empty BIG for {drug_name} — no complementary pharmacophore "
              f"pairs found within the binding site. Check pdb_path and binding site detection.")
        continue

    # Map graph nodes to qubit indices
    node2idx  = {n: k for k, n in enumerate(G.nodes())}
    idx2node  = {k: n for n, k in node2idx.items()}
    nodes     = list(idx2node.keys())
    qubit_num = len(nodes)
    edges     = [(node2idx[u], node2idx[v]) for u, v in G.edges()]
    non_edges = [(u, v) for u, v in itertools.combinations(nodes, 2)
                 if (u,v) not in edges and (v,u) not in edges]
    weights   = [1.0 / np.linalg.norm(
                     lig_feats[idx2node[k][0]]["pos"] - rec_feats[idx2node[k][1]]["pos"])
                 for k in nodes]
    penalty   = 1.2 * max(weights)

    # Hamiltonian + Pauli decomposition
    ham   = ham_clique(penalty, nodes, weights, non_edges)
    coef  = term_coefficients(ham)
    words = term_words(ham, qubit_num)

    # Classical optimiser
    np.random.seed(13)
    cudaq.set_random_seed(13)
    parameter_count = (2 * qubit_num + len(coef)) * NUM_LAYERS
    optimizer = cudaq.optimizers.NelderMead()
    optimizer.initial_parameters = np.random.uniform(-np.pi/8, np.pi/8, parameter_count)

    # Capture loop variables by value so the closure is correct each iteration
    def objective(params, _h=ham, _qn=qubit_num, _nl=NUM_LAYERS, _c=coef, _w=words):
        return cudaq.observe(dc_qaoa, _h, _qn, _nl, params, _c, _w).expectation()

    optimal_expectation, optimal_parameters = optimizer.optimize(
        dimensions=parameter_count, function=objective)

    binding_score = -optimal_expectation
    binding_scores.append(binding_score)
    ligand_smiles_list.append(smiles)
    ligand_names_list.append(drug_name)
    print(f"  Binding score: {binding_score:.6f}")

# ── Save for ADMET notebook ───────────────────────────────────────────────────
_out = {
    "binding_scores":     binding_scores,
    "ligand_smiles_list": ligand_smiles_list,
    "ligand_names_list":  ligand_names_list,
}
with open('/content/docking_results.json', 'w') as _f:
    json.dump(_out, _f)

print(f"\n{'='*60}")
print(f"Docking complete: {len(binding_scores)}/{len(CANDIDATE_LIGANDS)} ligands processed")
print(f"Results saved → /content/docking_results.json")
print(f"{'='*60}")
for name, score in zip(ligand_names_list, binding_scores):
    print(f"  {name:<15s}  {score:.6f}")

## 4. ADMET Profiling + Final Ranking

In [ ]:
# ── ADMET profiling + final ranking ─────────────────────────────────────────
model    = ADMETModel()
admet_df = model.predict(smiles=ligand_smiles_list)

admet_df["lipinski_pass"] = [lipinski_ok(Chem.MolFromSmiles(s)) for s in ligand_smiles_list]

good_admet = (
    (admet_df["HIA_Hou"]      == True)  &
    (admet_df["BBB_Martins"]  == False) &
    (admet_df["hERG"]         == False) &
    (admet_df["CYP3A4_Veith"] == False) &
    (admet_df["lipinski_pass"])
)
admet_df["admet_pass"]    = good_admet
admet_df["drug_name"]     = ligand_names_list
admet_df["binding_score"] = binding_scores

rank_df = (admet_df
           .sort_values(["admet_pass","binding_score"], ascending=[False,False])
           .reset_index(drop=True))

print("\n═══ FINAL RANKED CANDIDATES ══════════════════════════════════")
display_cols = ["drug_name","binding_score","HIA_Hou","BBB_Martins",
                "hERG","CYP3A4_Veith","lipinski_pass","admet_pass"]
print(rank_df[display_cols].to_string(index=True))

print("\n── Top recommendation ──")
top = rank_df[rank_df["admet_pass"]==True]
if len(top):
    best = top.iloc[0]
    print(f"  {best['drug_name']}  (binding score {best['binding_score']:.4f})")
else:
    print("  No ligand passed all ADMET filters. Best by binding score:")
    best = rank_df.iloc[0]
    print(f"  {best['drug_name']}  (binding score {best['binding_score']:.4f})")